# 02. Agent Memory and Recall

Prerequisite: run `01_ingest_and_deduplicate.ipynb` first so this notebook can reuse the populated FalkorDB graph.

Sample data note:

- The tutorial texts in `notebooks/experimental_data/` are Medium.com articles by Filip Wojcik.
- Source profile: `https://medium.com/@filip.igor.wojcik`
- They are fully accessible without a subscription.

Install prerequisites before running the notebook:

```bash
uv sync --group falkordb --extra notebooks
```


In [24]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import autoroot  # noqa
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext
from rich import print

from grawiki.db import FalkorGraphDB
from grawiki.rag import GraphRAG

load_dotenv()


def locate_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "experimental_data").exists():
        return cwd
    if (cwd / "notebooks" / "experimental_data").exists():
        return cwd / "notebooks"
    raise FileNotFoundError(
        "Could not locate the notebooks directory from the current working directory."
    )


NOTEBOOKS_DIR = locate_notebooks_dir()
DB_PATH = NOTEBOOKS_DIR / "local_falcor.db"
MODEL = (
    os.getenv("GRAWIKI_MODEL")
    or os.getenv("LLM_MODEL")
    or "openrouter/openai/gpt-5-mini"
)

AGENT_MODEL = "openrouter:openai/gpt-5-mini"

EMBEDDING_MODEL = (
    os.getenv("GRAWIKI_EMBEDDING_MODEL")
    or os.getenv("EMBEDDING_MODEL")
    or "openrouter:openai/text-embedding-3-small"
)

database = FalkorGraphDB(db_path=DB_PATH, graph_name="grawiki_notebooks")
rag = GraphRAG(model=MODEL, embedding_model=EMBEDDING_MODEL, db=database)


@dataclass
class NotebookDeps:
    rag: GraphRAG


def node_payload(node) -> dict[str, Any]:
    payload = node.model_dump()
    payload["labels"] = sorted(node.labels)
    return payload


def hit_payload(hit) -> dict[str, Any]:
    return {
        "score": round(hit.score, 4),
        "matched_on": hit.matched_on,
        "node": node_payload(hit.node),
    }


deps = NotebookDeps(rag=rag)
USER_ID = "demo-user"

In [25]:
agent = Agent(
    model=AGENT_MODEL,
    deps_type=NotebookDeps,
    system_prompt=(
        "You are a notebook demo agent for GraWiki. Use search_knowledge_graph for facts from the graph, "
        "remember_user_fact when the user tells you something durable about themselves, and "
        "recall_user_memories before answering user-specific follow-up questions.",
        "You are assiting user with id = 'demo-user'"
    ),
)


@agent.tool
async def search_knowledge_graph(
    ctx: RunContext[NotebookDeps],
    query: str,
    limit: int = 5,
) -> list[dict[str, Any]]:
    hits = await ctx.deps.rag.search(query, limit=limit)
    return [hit_payload(hit) for hit in hits]


@agent.tool
async def remember_user_fact(
    ctx: RunContext[NotebookDeps],
    memory_text: str,
    user_id: str,
    related_node_ids: list[str] | None = None,
) -> dict[str, Any]:
    memory = await ctx.deps.rag.remember(
        memory_text,
        metadata={"user_id": user_id},
        related_node_ids=related_node_ids or [],
    )
    return node_payload(memory)


@agent.tool
async def recall_user_memories(
    ctx: RunContext[NotebookDeps],
    query: str,
    user_id: str,
    hops: int = 1,
    limit: int = 5,
) -> list[dict[str, Any]]:
    hits = await ctx.deps.rag.recall(query, user_id=user_id, hops=hops, limit=limit)
    return [hit_payload(hit) for hit in hits]


print("Agent and tools are ready.")

Agent and tools are ready.

## Demo conversation flow

The first call asks the agent to answer a factual question about the graph built in notebook 1.


In [26]:
factual_result = await agent.run(
    "Using the knowledge graph, summarize the main idea behind connecting LLMs and knowledge graphs.",
    deps=deps,
)
print(factual_result.output)

Brief summary — main idea

- Problem: LLMs are powerful at generating fluent text but reason probabilistically from training data and can 
hallucinate or be uncertain about factual details.  
- KG role: Knowledge graphs (KGs) encode structured, explicit facts as entities, relations, types and schemas 
(heterogeneous/property graphs) that capture domain knowledge in a machine-readable form.  
- Main idea: Combine LLMs’ language and pattern capabilities with KGs’ structured factual grounding so the system 
keeps LLM fluency while gaining correctness, explainability, and explicit relational reasoning.  

Why this helps
- Reduces hallucination by grounding LLM outputs in explicit KG facts.  
- Enables relational and multi-hop reasoning over entities (via KG queries or graph neural networks).  
- Improves domain accuracy and traceability: answers can point to specific KG nodes/relations.  
- Supports retrieval-augmented workflows: retrieve relevant KG subgraphs, convert to context/embeddings, then 
condition the LLM.

Common integration patterns
- Context injection / retrieval-augmented generation: fetch KG facts, add them to the LLM prompt or to a retrieval 
index used at inference.  
- Training/enrichment: incorporate KG-derived signals or entity embeddings into model training or fine-tuning.  
- Hybrid pipelines: use symbolic KG queries (SPARQL/Cypher) to get precise facts, and use LLMs to interpret, 
naturalize, or reason with those facts.  
- Graph neural networks: encode graph structure into vector embeddings that LLMs or downstream modules use for 
reasoning or ranking.

Practical tools & considerations
- Tools: graph DBs (Neo4j), graph ML libs (PyTorch Geometric), open KGs (DBpedia) are commonly used.  
- Challenges: KG coverage/incompleteness, schema alignment with LLMs’ internal representations, latency of KG 
queries, and designing prompts/architectures that properly combine symbolic and neural signals.

Short example pipeline
- Query arrives → retrieve relevant KG subgraph → convert facts/embeddings into LLM context or input features → LLM
generates answer constrained/validated against KG → optionally update KG with verified results.

If you want, I can:  
- sketch a concrete retrieval+prompt template for grounding an LLM with KG facts, or  
- show a short example combining a Neo4j query with a prompt for an LLM. Which would be most helpful?

In [27]:
remember_result = await agent.run(
    """My name is Filip and I work as a researcher and data scientist. 
    I focus mostly on knowledge graphs and their integration with large language models.""",
    deps=deps,
)
print(remember_result.output)

Thanks — I’ve saved that. I’ll remember you’re Filip, a researcher and data scientist who focuses on knowledge 
graphs and their integration with large language models.

Would you like me to remember anything else (pronouns, current projects, preferred tools, timelines)? Or should I 
help with something right now on KG ↔ LLM integration?

In [30]:
follow_up_result = await agent.run(
    "What do you remember about Filip and his interests?",
    deps=deps,
)
print(follow_up_result.output)

I remember that Filip is a researcher and data scientist who focuses on knowledge graphs and their integration with
large language models.  

Would you like me to add or update anything about his interests or background?

## Direct recall inspection

The facade-level recall API returns memory hits with readable graph context stored in `node.properties['recall_context']`.


In [32]:
recall_hits = await rag.recall("Filip", hops=2, limit=5)
for hit in recall_hits:
    payload = hit_payload(hit)
    payload["recall_context"] = hit.node.properties.get("recall_context", "")
    print(payload)

{
    'score': 0.4536,
    'matched_on': 'vector',
    'node': {
        'id': '1c270f93-16ca-450b-a7cd-426e4c05aefb',
        'labels': ['__memory__'],
        'semantic_key': '1c270f93-16ca-450b-a7cd-426e4c05aefb',
        'name': 'User is Filip, a researcher and data scientist focusing on knowledge graphs and ',
        'properties': {
            'recall_context': 'User is Filip, a researcher and data scientist focusing on knowledge graphs and  
-[__mentions__]-> Data Scientist\nUser is Filip, a researcher and data scientist focusing on knowledge graphs and  
-[__mentions__]-> Filip\nUser is Filip, a researcher and data scientist focusing on knowledge graphs and  
-[__mentions__]-> Knowledge graph — LLM integration\nUser is Filip, a researcher and data scientist focusing on 
knowledge graphs and  -[__mentions__]-> Knowledge graphs\nUser is Filip, a researcher and data scientist focusing 
on knowledge graphs and  -[__mentions__]-> Large language models'
        },
        'embedding': [],
        'content': 'User is Filip, a researcher and data scientist focusing on knowledge graphs and their 
integration with large language models.',
        'creation_date': '2026-04-30T11:14:25.061673',
        'metadata': {'user_id': 'demo-user'}
    },
    'recall_context': 'User is Filip, a researcher and data scientist focusing on knowledge graphs and  
-[__mentions__]-> Data Scientist\nUser is Filip, a researcher and data scientist focusing on knowledge graphs and  
-[__mentions__]-> Filip\nUser is Filip, a researcher and data scientist focusing on knowledge graphs and  
-[__mentions__]-> Knowledge graph — LLM integration\nUser is Filip, a researcher and data scientist focusing on 
knowledge graphs and  -[__mentions__]-> Knowledge graphs\nUser is Filip, a researcher and data scientist focusing 
on knowledge graphs and  -[__mentions__]-> Large language models'
}

In [33]:
memory_rows = database.ro_query(
    "MATCH (m:__memory__) OPTIONAL MATCH (m)-[r]->(n) RETURN m.id, m.name, type(r), n.id, n.name ORDER BY m.creation_date, m.id LIMIT 20"
).result_set
print({"memory_rows": memory_rows})

{
    'memory_rows': [
        [
            '1c270f93-16ca-450b-a7cd-426e4c05aefb',
            'User is Filip, a researcher and data scientist focusing on knowledge graphs and ',
            '__mentions__',
            'e9ae9c68-9663-4493-b83e-d625f1e8c90a',
            'Knowledge graphs'
        ],
        [
            '1c270f93-16ca-450b-a7cd-426e4c05aefb',
            'User is Filip, a researcher and data scientist focusing on knowledge graphs and ',
            '__mentions__',
            '80796553-0d71-486c-9f4d-aeb1be31154d',
            'Large language models'
        ],
        [
            '1c270f93-16ca-450b-a7cd-426e4c05aefb',
            'User is Filip, a researcher and data scientist focusing on knowledge graphs and ',
            '__mentions__',
            'cbfc2e2f-d7bf-4c25-ba3a-8527f579d8c5',
            'Filip'
        ],
        [
            '1c270f93-16ca-450b-a7cd-426e4c05aefb',
            'User is Filip, a researcher and data scientist focusing on knowledge graphs and ',
            '__mentions__',
            'b5679481-1aee-41f4-90a0-c0f62f2dc6d3',
            'Data Scientist'
        ],
        [
            '1c270f93-16ca-450b-a7cd-426e4c05aefb',
            'User is Filip, a researcher and data scientist focusing on knowledge graphs and ',
            '__mentions__',
            '1824cac7-260b-4529-9b1d-59657a417f14',
            'Researcher'
        ],
        [
            '1c270f93-16ca-450b-a7cd-426e4c05aefb',
            'User is Filip, a researcher and data scientist focusing on knowledge graphs and ',
            '__mentions__',
            'f4bb38ed-cd66-4af4-8eee-54c08954a644',
            'Knowledge graph — LLM integration'
        ]
    ]
}

# Reintegrate entities after memory saving session

In [34]:
applied_reports = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.95, dry_run=False
)
entities_after_dedupe = await database.list_entities()

print({"entity_count_after_dedupe": len(entities_after_dedupe)})
for entity in entities_after_dedupe[:10]:
    print(
        {
            "id": entity.id,
            "name": entity.name,
            "labels": sorted(entity.labels),
            "semantic_key": entity.semantic_key,
        }
    )

{'entity_count_after_dedupe': 77}

{
    'id': 'bd26e330-e401-436a-9bce-33dc84cc1bd0',
    'name': 'Researchers',
    'labels': ['Agent'],
    'semantic_key': 'Agent_researchers'
}

{
    'id': '4a9c9272-1013-4dac-89fc-a2974a03c274',
    'name': 'Planned series',
    'labels': ['Collection'],
    'semantic_key': 'Collection_planned-series'
}

{
    'id': '25783d96-444e-45b4-9a54-920b73bfeae7',
    'name': 'Corpus',
    'labels': ['Concept'],
    'semantic_key': 'Concept_corpus'
}

{
    'id': '381565a6-8dce-4dde-809b-ef506f2b0837',
    'name': 'Counterfactual reasoning',
    'labels': ['Concept'],
    'semantic_key': 'Concept_counterfactual-reasoning'
}

...

{
    'id': 'd7a36690-accd-47a6-8d14-e0863a1880c3',
    'name': 'graph neural networks',
    'labels': ['Concept'],
    'semantic_key': 'Concept_graph-neural-networks'
}

{
    'id': '798f187d-f8d8-48de-9079-f9b6f0f1c4bb',
    'name': 'Heterogeneous graph (HG)',
    'labels': ['Concept'],
    'semantic_key': 'Concept_heterogeneous-graph'
}

{
    'id': '3cb27674-b6ec-44ea-9641-a496d488a400',
    'name': 'heterogeneous graphs',
    'labels': ['Concept'],
    'semantic_key': 'Concept_heterogeneous-graphs'
}

{
    'id': '7f146735-551a-44ed-bf5e-21598800abf2',
    'name': 'KG reasoning (KGR)',
    'labels': ['Concept'],
    'semantic_key': 'Concept_kg-reasoning'
}

{
    'id': '93f0f8e5-f3ae-42b6-9cae-7eef58e481ea',
    'name': 'Knowledge graphs',
    'labels': ['Concept'],
    'semantic_key': 'Concept_knowledge-graphs'
}

In [35]:
# Run this once when you are finished with the notebook session.
database.close()